# Exercises XP: Text Preprocessing, NER, POS, and Word2Vec

Use this guided notebook to follow the platform instructions step by step. Prefilled cells are ready to run; cells marked TODO expect your code or analysis.

## What you will learn
- Clean and normalize raw reviews with tokenization, stopword removal, and lemmatization.
- Extract linguistic features with named entity recognition (NER) and part-of-speech (POS) tagging.
- Train a simple Word2Vec model and interpret its vector dimensions.
- Visualize word embeddings to reason about semantic neighborhoods.

## What you will create
- A `preprocess_text` function that lowercases, strips punctuation, removes stopwords, and lemmatizes.
- `perform_ner` and `perform_pos_tagging` helpers to analyze raw vs cleaned text.
- A Word2Vec model plus a helper to plot embeddings for inspection.

> **Learning point**  
> Run the setup cells once, then progress through each exercise sequentially. Print intermediate results to verify every helper works before moving on.

## Setup · install libraries
Run once to install spaCy, nltk, gensim, and plotting utilities.

In [ ]:
%pip install --quiet spacy nltk gensim matplotlib seaborn --upgrade

In [ ]:
import nltk
from spacy.cli import download as spacy_download
import spacy

resources = [
    "punkt",
    "punkt_tab",
    "wordnet",
    "omw-1.4",
    "stopwords",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
    "tagsets",
]
for res in resources:
    nltk.download(res, quiet=True)

spacy_download("en_core_web_sm")

nlp = spacy.load("en_core_web_sm")
print("spaCy pipeline:", nlp.pipe_names)

---
## Exercise 1 · Explore text preprocessing, NER, and POS tags

Here is the dataset you will reuse in every step.

In [ ]:
data = {
    'Review': [
        "At McDonald's the food was ok and the service was bad.",
        "I would not recommend this Japanese restaurant to anyone.",
        "I loved this restaurant when I traveled to Thailand last summer.",
        "The menu of Loving has a wide variety of options.",
        "The staff was friendly and helpful at Google's employees restaurant.",
        "The ambiance at Bella Italia is amazing, and the pasta dishes are delicious.",
        "I had a terrible experience at Pizza Hut. The pizza was burnt, and the service was slow.",
        "The sushi at Sushi Express is always fresh and flavorful.",
        "The steakhouse on Main Street has a cozy atmosphere and excellent steaks.",
        "The dessert selection at Sweet Treats is to die for!"
    ]
}
raw_reviews = data['Review']
raw_reviews

### 1.1 Build `preprocess_text()`
Create a function that:
1. Lowercases and tokenizes text.
2. Removes punctuation tokens.
3. Removes English stopwords.
4. Applies a lemmatizer.
5. Returns the cleaned string joined by spaces.

Print the processed reviews to confirm every stage works.

In [ ]:
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def preprocess_text(text: str) -> str:
    """Lowercase, tokenize, strip punctuation, drop stopwords, and lemmatize a review."""
    # Step 1 — Lowercase and tokenize
    tokens = word_tokenize(text.lower())

    # Step 2 — Remove punctuation tokens
    tokens = [t for t in tokens if t not in string.punctuation]

    # Step 3 — Remove stopwords
    tokens = [t for t in tokens if t not in stop_words]

    # Step 4 — Lemmatize remaining tokens
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    # Step 5 — Join back into a single cleaned string
    return " ".join(tokens)


# Quick sanity-check on a single review
sample = raw_reviews[0]
print("Original :", sample)
print("Processed:", preprocess_text(sample))

### 1.2 Create a cleaned dataset
Apply `preprocess_text` to every review and keep both raw and cleaned versions side by side.

In [ ]:
# Apply preprocess_text to every review
cleaned_reviews = [preprocess_text(review) for review in raw_reviews]

# Display both versions side by side
for i, (raw, cleaned) in enumerate(zip(raw_reviews, cleaned_reviews), 1):
    print(f"--- Review {i} ---")
    print(f"RAW    : {raw}")
    print(f"CLEANED: {cleaned}")
    print()

### 1.3 Named Entity Recognition (NER)
Create `perform_ner(text)` that returns `(entity, label_)` pairs using `en_core_web_sm`. Test it on a few reviews.

In [ ]:
def perform_ner(text: str):
    """Return (entity_text, label) pairs found by spaCy's en_core_web_sm pipeline."""
    # Run the spaCy pipeline on the text
    doc = nlp(text)
    # Collect each entity's surface form and its semantic label
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    return entities


# Test on the first three raw reviews
print("=== NER on raw reviews ===")
for i, review in enumerate(raw_reviews[:3], 1):
    ents = perform_ner(review)
    print(f"Review {i}: {review}")
    print(f"Entities: {ents}")
    print()

### 1.4 Part-of-Speech tagging (POS)
Create `perform_pos_tagging(text)` using `nltk.pos_tag`. Test it on both raw and cleaned text.

Use `nltk.help.upenn_tagset('NN')` to recall tag meanings if needed.

In [ ]:
from nltk import pos_tag, word_tokenize

def perform_pos_tagging(text: str):
    """Return POS tags for a given text using the Penn Treebank tagset."""
    # Tokenize the text first
    tokens = word_tokenize(text)
    # Apply NLTK's averaged perceptron POS tagger
    tags = pos_tag(tokens)
    return tags


# Quick test on the first review
print("POS tags (raw)   :", perform_pos_tagging(raw_reviews[0]))
print()
print("POS tags (cleaned):", perform_pos_tagging(cleaned_reviews[0]))

# Reference: meaning of common tags
print("\n--- Tag reference ---")
for tag in ["NN", "NNP", "VBD", "JJ", "RB"]:
    nltk.help.upenn_tagset(tag)

### 1.5 Apply NER and POS on raw vs cleaned text
Compare outputs on the same entries to see how preprocessing affects tagging.

In [ ]:
sample_indices = [0, 1]  # first two reviews

# ── NER on raw text ──────────────────────────────────────────────────
print("=" * 60)
print("NER on RAW text")
print("=" * 60)
for i in sample_indices:
    print(f"\nReview {i+1}: {raw_reviews[i]}")
    print("Entities:", perform_ner(raw_reviews[i]))

# ── NER on cleaned text ───────────────────────────────────────────────
print("\n" + "=" * 60)
print("NER on CLEANED text")
print("=" * 60)
for i in sample_indices:
    print(f"\nReview {i+1}: {cleaned_reviews[i]}")
    print("Entities:", perform_ner(cleaned_reviews[i]))

# ── POS on raw text ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print("POS tags on RAW text")
print("=" * 60)
for i in sample_indices:
    print(f"\nReview {i+1}: {raw_reviews[i]}")
    print("POS tags:", perform_pos_tagging(raw_reviews[i]))

# ── POS on cleaned text ───────────────────────────────────────────────
print("\n" + "=" * 60)
print("POS tags on CLEANED text")
print("=" * 60)
for i in sample_indices:
    print(f"\nReview {i+1}: {cleaned_reviews[i]}")
    print("POS tags:", perform_pos_tagging(cleaned_reviews[i]))

### Analysis — NER & POS: raw vs cleaned

| Aspect | Raw text | Cleaned text |
|--------|----------|--------------|
| **NER** | Entities like `McDonald's` (ORG), `Thailand` (GPE), `last summer` (DATE) are correctly detected because proper nouns and their context are preserved. | Most entities **disappear** — stopword removal strips context words (e.g., *"last"*) and lemmatization can distort proper nouns, making them unrecognizable to spaCy. |
| **POS** | Rich tag variety: `NNP` (proper nouns), `VBD` (past-tense verbs), `JJ` (adjectives), `IN` (prepositions). | Reduced tag set: mostly `NN`, `JJ`, `VB`. Determiners, prepositions, and auxiliaries are gone — grammatical structure is lost. |

**Takeaway:** Apply NER and POS tagging **before** preprocessing to preserve linguistic structure. Use cleaned text only for downstream tasks like Word2Vec or bag-of-words classification.

---
## Exercise 2 · Plotting word embeddings

### 2.1 Train a Word2Vec model
Vectorize the preprocessed/tokenized dataset with `Word2Vec` from `gensim.models`. Reuse the cleaned text and adjust parameters like `vector_size`, `window`, and `sg`.

In [ ]:
from gensim.models import Word2Vec

# Tokenize cleaned reviews on whitespace (already cleaned in Exercise 1)
tokenized_reviews = [review.split() for review in cleaned_reviews]

print("Tokenized reviews (sample):")
for i, tokens in enumerate(tokenized_reviews[:3], 1):
    print(f"  Review {i}: {tokens}")

# Train the Word2Vec model
# - vector_size : dimensionality of the word vectors
# - window      : max distance between the current and predicted word within a sentence
# - min_count   : ignores words with frequency lower than this
# - sg          : 0 = CBOW (uses context to predict word), 1 = Skip-Gram (uses word to predict context)
# - epochs      : number of iterations over the corpus
w2v_model = Word2Vec(
    sentences=tokenized_reviews,
    vector_size=50,   # small corpus → small vectors
    window=3,
    min_count=1,      # keep all words (tiny dataset)
    sg=0,             # CBOW
    epochs=100,
    seed=42
)

print("\nWord2Vec model trained successfully:", w2v_model)

### 2.2 Inspect embedding dimensions
Print and interpret the vector size and vocabulary size from the fitted model.

In [ ]:
vocab       = list(w2v_model.wv.key_to_index.keys())
vocab_size  = len(vocab)
vector_size = w2v_model.vector_size

print(f"Vocabulary size : {vocab_size} unique words")
print(f"Vector size     : {vector_size} dimensions per word")
print(f"\nAll words in vocabulary:")
print(vocab)

# Show the embedding vector for a sample word
sample_word = "restaurant"
if sample_word in w2v_model.wv:
    vec = w2v_model.wv[sample_word]
    print(f"\nVector for '{sample_word}' (first 10 dims): {vec[:10].round(4)}")

print("""
Interpretation
--------------
Each word is represented as a dense vector of {size} floating-point numbers.
Each dimension encodes a latent semantic or syntactic feature learned from context.
Words appearing in similar contexts will have similar vectors (small cosine distance).
With only 10 reviews, the vectors are noisy — a larger corpus yields richer geometry.
""".format(size=vector_size))

### 2.3 Plot word embeddings
Complete `plot_word_embeddings(model)` to scatter-plot the first two dimensions of the learned vectors and annotate each point with its word. Discuss whether related words cluster together.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_word_embeddings(model, words=None, title="Word2Vec Embeddings (dim 0 vs dim 1)"):
    """
    Scatter-plot the first two raw dimensions of Word2Vec vectors.

    Args:
        model : trained gensim Word2Vec model
        words : list of words to plot (defaults to entire vocabulary)
        title : plot title
    """
    if words is None:
        words = list(model.wv.key_to_index.keys())

    # Collect 2-D coordinates (first two raw dimensions)
    coords = np.array([model.wv[w] for w in words])
    xs = coords[:, 0]
    ys = coords[:, 1]

    # Draw scatter plot
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.scatter(xs, ys, s=60, color="steelblue", alpha=0.8, edgecolors="white", linewidth=0.6)

    # Annotate each point with the corresponding word
    for word, x, y in zip(words, xs, ys):
        ax.annotate(
            word,
            xy=(x, y),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=9,
            color="#2c3e50"
        )

    ax.set_title(title, fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Dimension 0", fontsize=11)
    ax.set_ylabel("Dimension 1", fontsize=11)
    ax.axhline(0, color="lightgray", linewidth=0.8, linestyle="--")
    ax.axvline(0, color="lightgray", linewidth=0.8, linestyle="--")
    ax.grid(True, linestyle=":", alpha=0.5)
    plt.tight_layout()
    plt.savefig("word2vec_embeddings.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Plot saved → word2vec_embeddings.png")


# Plot the full vocabulary
plot_word_embeddings(w2v_model)

### Analysis — Word embeddings plot

**Are related words close together?**  
With only 10 reviews (~60 unique tokens after preprocessing), the corpus is too small for Word2Vec to learn reliable geometry. A few patterns may emerge by chance (e.g., food-related words grouping loosely), but the signal-to-noise ratio is low.

**Why does clustering fail on small data?**  
- Word2Vec needs many co-occurrence examples to learn that `pizza` and `sushi` are both *food items*.
- With `min_count=1`, rare words (appearing only once) have poorly initialized vectors.
- The first two raw dimensions do not necessarily capture the most variance — PCA or t-SNE would give a richer view.

**Expected clusters on a large dataset:**  
- Place names: `Thailand`, `italy`, `street` …  
- Food items: `pizza`, `sushi`, `pasta`, `steak`, `dessert` …  
- Sentiment words: `terrible`, `amazing`, `delicious`, `bad` …

### 2.4 Go further — PCA visualization & hyperparameter experiments

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np


def plot_pca_embeddings(model, words=None, n_components=2,
                        title="Word2Vec Embeddings — PCA projection"):
    """
    Project Word2Vec vectors to 2-D with PCA and scatter-plot them.
    PCA captures the directions of maximum variance, giving a cleaner view
    than plotting two raw arbitrary dimensions.
    """
    if words is None:
        words = list(model.wv.key_to_index.keys())

    vectors = np.array([model.wv[w] for w in words])

    # Reduce to 2 dimensions
    pca     = PCA(n_components=n_components, random_state=42)
    reduced = pca.fit_transform(vectors)
    expl    = pca.explained_variance_ratio_

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.scatter(reduced[:, 0], reduced[:, 1],
               s=70, color="darkorange", alpha=0.85,
               edgecolors="white", linewidth=0.6)

    for word, (x, y) in zip(words, reduced):
        ax.annotate(word, xy=(x, y), xytext=(5, 5),
                    textcoords="offset points", fontsize=9, color="#2c3e50")

    ax.set_title(
        f"{title}\nPC1={expl[0]:.1%} variance | PC2={expl[1]:.1%} variance",
        fontsize=12, fontweight="bold", pad=12
    )
    ax.set_xlabel(f"PC1 ({expl[0]:.1%} variance)", fontsize=11)
    ax.set_ylabel(f"PC2 ({expl[1]:.1%} variance)", fontsize=11)
    ax.axhline(0, color="lightgray", linewidth=0.8, linestyle="--")
    ax.axvline(0, color="lightgray", linewidth=0.8, linestyle="--")
    ax.grid(True, linestyle=":", alpha=0.5)
    plt.tight_layout()
    plt.savefig("word2vec_pca.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("PCA plot saved → word2vec_pca.png")


plot_pca_embeddings(w2v_model)

In [ ]:
# ── Hyperparameter comparison ──────────────────────────────────────────
# Train three variants and compare vocabulary / vector sizes

configs = [
    {"vector_size": 50,  "window": 2, "sg": 0, "label": "CBOW  window=2 dim=50"},
    {"vector_size": 50,  "window": 5, "sg": 1, "label": "SG    window=5 dim=50"},
    {"vector_size": 100, "window": 3, "sg": 0, "label": "CBOW  window=3 dim=100"},
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, cfg in zip(axes, configs):
    m = Word2Vec(
        sentences=tokenized_reviews,
        vector_size=cfg["vector_size"],
        window=cfg["window"],
        min_count=1,
        sg=cfg["sg"],
        epochs=100,
        seed=42
    )
    words   = list(m.wv.key_to_index.keys())
    vecs    = np.array([m.wv[w] for w in words])
    pca     = PCA(n_components=2, random_state=42)
    reduced = pca.fit_transform(vecs)

    ax.scatter(reduced[:, 0], reduced[:, 1],
               s=55, color="steelblue", alpha=0.8, edgecolors="white")
    for word, (x, y) in zip(words, reduced):
        ax.annotate(word, xy=(x, y), xytext=(4, 4),
                    textcoords="offset points", fontsize=7)
    ax.set_title(cfg["label"], fontsize=10, fontweight="bold")
    ax.grid(True, linestyle=":", alpha=0.4)

plt.suptitle("Hyperparameter comparison — PCA of Word2Vec embeddings",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("word2vec_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Comparison plot saved → word2vec_comparison.png")

---
## Summary

| Step | Key function | Library | Output |
|------|-------------|---------|--------|
| Preprocessing | `preprocess_text()` | NLTK | Lowercase, no stopwords, lemmatized string |
| NER | `perform_ner()` | spaCy | `[(entity, label)]` pairs |
| POS tagging | `perform_pos_tagging()` | NLTK | `[(token, tag)]` pairs |
| Word embeddings | `Word2Vec` | Gensim | Dense vectors per word |
| Visualization | `plot_word_embeddings()` / `plot_pca_embeddings()` | Matplotlib / scikit-learn | 2-D scatter plots |

### Key takeaways
1. **Preprocessing order matters** — run NER and POS on raw text before cleaning to preserve linguistic signals.
2. **Small corpora ≠ good embeddings** — Word2Vec needs thousands of sentences to learn meaningful geometry.
3. **PCA > raw dimensions** — projecting with PCA captures maximum variance, giving a more informative scatter plot than two arbitrary raw dimensions.
4. **CBOW vs Skip-Gram** — CBOW is faster and works better with frequent words; Skip-Gram handles rare words better and generally yields richer vectors on small corpora.